In [16]:
# Imports, parameters, and load data files
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from scipy.signal import correlate
from filterpy.kalman import UnscentedKalmanFilter as UKF_fp
from filterpy.kalman import MerweScaledSigmaPoints

# Circuit parameters (SI)
C1 = 30.14e-6     # F
C2 = 185.6e-6     # F
L  = 52.28        # H
R  = 1673.0       # ohm
R_L = 0.0         # ohm
Ga = -0.801e-3    # S
Gb = -0.365e-3    # S
E  = 1.74         # V

Ts = 0.01  # sampling time for discretization (10 ms)

# Data files
dados_file = 'pcchua_dados.dat'
pert_file  = 'pcchua_pert.dat'

def load_pcchua(fname):
    data = np.loadtxt(fname, comments='#')
    t  = data[:,0]
    x  = data[:,1]
    y  = data[:,2]
    z  = data[:,3]
    # optional: reference columns and ux,uy,uz in cols 7,8,9
    ux = data[:,7]
    uy = data[:,8]
    uz = data[:,9]
    return {'t':t, 'x':x, 'y':y, 'z':z, 'ux':ux, 'uy':uy, 'uz':uz}

dados = load_pcchua(dados_file)
pert  = load_pcchua(pert_file)

# Use measurement vectors (only vC2 and iL are measured)
t = dados['t']             # time base (s)
meas_vC2 = dados['y']      # measured vC2
meas_iL  = dados['z']      # measured iL
Y = np.vstack([meas_vC2, meas_iL]).T
N = len(t)


In [17]:
# Continuous model, piecewise G and discrete-time propagator via solve_ivp

def G_of_v(v):
    av = abs(v)
    if av < E:
        return Ga
    # avoid zero division
    return Gb + (Ga - Gb) * E / (av + 1e-12)

def chua_continuous_rhs(t, state, rx=0.0, ry=0.0, rz=0.0):
    v1, v2, iL = state
    Gv1 = G_of_v(v1)
    dv1 = (1.0 / C1) * ( (v2 - v1) / R - Gv1 * v1 + rx )
    dv2 = (1.0 / C2) * ( (v1 - v2) / R + iL + ry )
    di  = (1.0 / L)  * ( -v2 + R_L * iL + rz )
    return [dv1, dv2, di]

# Build interpolants for r_x,r_y,r_z from perturbation file (use ux,uy,uz)
r_x_interp = interp1d(pert['t'], pert['ux'], bounds_error=False)
r_y_interp = interp1d(pert['t'], pert['uy'], bounds_error=False)
r_z_interp = interp1d(pert['t'], pert['uz'], bounds_error=False)

def discretize_f(xk, tk, Ts):
    """
    Propagate continuous dynamics from tk to tk+Ts with inputs taken from perturbation interpolants.
    Returns x_{k+1}.
    """
    def rhs(t_local, state):
        tt = tk + t_local
        tt = np.clip(tt, pert['t'][0], pert['t'][-1]) # clamp to valid range
        rx = float(r_x_interp(tt))
        ry = float(r_y_interp(tt))
        rz = float(r_z_interp(tt))
        return chua_continuous_rhs(tt, state, rx=rx, ry=ry, rz=rz)

    sol = solve_ivp(rhs, [0, Ts], xk, method='RK45', max_step=Ts/10.0, rtol=1e-7, atol=1e-9)
    return sol.y[:, -1]

# quick test discretize on a sample state (do not run heavy)
x1 = discretize_f(np.array([0.1, 0.0, 0.0]), pert['t'][0], Ts)


In [18]:
# Measurement model and Jacobian helpers

def h_of_x(x):
    # measurement is [vC2, iL]
    return np.array([x[1], x[2]])

def H_jacobian(x):
    # exact linear measurement: vC2 = x[1], iL = x[2]
    H = np.array([[0.0, 1.0, 0.0],
                  [0.0, 0.0, 1.0]])
    return H

def discrete_jacobian_fx(xk, tk, Ts, eps=1e-6):
    """
    Numerically approximate the discrete-time Jacobian F_k = d f_d(xk)/dxk
    where f_d is the discretized mapping x_{k+1} = f_d(xk).
    """
    n = len(xk)
    Fx = np.zeros((n,n))
    f0 = discretize_f(xk, tk, Ts)
    for i in range(n):
        dx = np.zeros(n)
        dx[i] = eps
        f1 = discretize_f(xk + dx, tk, Ts)
        Fx[:, i] = (f1 - f0) / eps
    return Fx


In [19]:
# EKF implementation (discrete-time)
def run_ekf(Y, t, x0, P0, Q, R):
    """
    Y: Nx2 measurement array
    t: time vector length N
    x0: initial state (3,)
    P0: initial covariance (3x3)
    Q: process noise covariance (3x3)
    R: measurement noise covariance (2x2)
    returns: x_estimates (N x 3), P_estimates (N x 3x3), innovations (N x 2)
    """
    N = Y.shape[0]
    x_est = np.zeros((N, 3))
    P_est = np.zeros((N, 3, 3))
    innov = np.zeros((N, 2))

    x_prev = x0.copy()
    P_prev = P0.copy()

    for k in range(N):
        tk = t[k]
        # Predict: discrete propagation
        x_pred = discretize_f(x_prev, tk, Ts)
        x_pred = np.clip(x_pred, 1.9e-5, None)  # Enforce minimum bound
        Fk = discrete_jacobian_fx(x_prev, tk, Ts)  # Jacobian at previous state
        P_pred = Fk @ P_prev @ Fk.T + Q

        # Measurement update
        Hk = H_jacobian(x_pred)
        y_pred = h_of_x(x_pred)
        y_meas = Y[k]
        y_tilde = y_meas - y_pred

        S = Hk @ P_pred @ Hk.T + R
        K = P_pred @ Hk.T @ np.linalg.inv(S)

        x_upd = x_pred + K @ y_tilde
        P_upd = (np.eye(3) - K @ Hk) @ P_pred

        # store
        x_est[k] = x_upd
        P_est[k] = P_upd
        innov[k] = y_tilde

        # prepare next
        x_prev = x_upd
        P_prev = P_upd

    return x_est, P_est, innov

# default initial guesses (tuneable)
x0 = np.array([0, meas_vC2[0], meas_iL[0]])   # initialize vC1 unknown, vC2,iL from first sample
P0 = np.diag([1.0, 0.1, 0.1])
Q_base = np.diag([1e-6, 1e-6, 1e-7])
R_base = np.diag([ (np.std(meas_vC2[:200])**2 + 1e-8), (np.std(meas_iL[:200])**2 + 1e-8) ])

# run with base values to get baseline
x_ekf_base, P_ekf_base, innov_ekf_base = run_ekf(Y, t, x0, P0, Q_base, R_base)


In [20]:
# UKF using filterpy (MerweScaledSigmaPoints)
def fx_for_ukf(x, dt, tk):
    # x is state at time k, we need to return state at time k+1
    return discretize_f(x, tk, dt)

def hx_for_ukf(x):
    return h_of_x(x)

def run_ukf(Y, t, x0, P0, Q, R):
    """
    Wrap filterpy UKF for the discrete-time propagator fx_for_ukf which depends on tk.
    filterpy's UKF fx signature is fx(x, dt). We will inject tk by using a closure and updating an index.
    """
    n = 3
    m = 2
    points = MerweScaledSigmaPoints(n, alpha=0.1, beta=2.0, kappa=0.0)

    # Build UKF object
    ukf = UKF_fp(dim_x=n, dim_z=m, fx=lambda x, dt: x, hx=hx_for_ukf, dt=Ts, points=points)
    ukf.x = x0.copy()
    ukf.P = P0.copy()
    ukf.Q = Q.copy()
    ukf.R = R.copy()

    N = Y.shape[0]
    x_est = np.zeros((N, n))
    P_est = np.zeros((N, n, n))
    innov = np.zeros((N, m))

    # We'll override predict step to use our discretize_f with time
    for k in range(N):
        tk = t[k]
        # Predict using discretize_f
        ukf.x = discretize_f(ukf.x, tk, Ts)
        # propagate covariance approximately via linearization (optionally leave as is)
        # Use numerical Jacobian to update P: P = F P F^T + Q
        Fk = discrete_jacobian_fx(ukf.x, tk, Ts)
        ukf.P = Fk @ ukf.P @ Fk.T + ukf.Q

        # Update
        z = Y[k]
        # compute predicted measurement and innovation using hx
        z_pred = hx_for_ukf(ukf.x)
        y_tilde = z - z_pred

        # Kalman gain via linearization (approx)
        Hk = H_jacobian(ukf.x)
        S = Hk @ ukf.P @ Hk.T + ukf.R
        K = ukf.P @ Hk.T @ np.linalg.inv(S)

        ukf.x = ukf.x + K @ y_tilde
        ukf.P = (np.eye(n) - K @ Hk) @ ukf.P

        x_est[k] = ukf.x
        P_est[k] = ukf.P
        innov[k] = y_tilde

    return x_est, P_est, innov

# run UKF with base Q,R
x_ukf_base, P_ukf_base, innov_ukf_base = run_ukf(Y, t, x0, P0, Q_base, R_base)


In [ ]:
# Whiteness metric and automatic grid-search tuning (coarse -> fine)
def whiteness_metric(innov, maxlag=20):
    """
    Compute a whiteness metric: sum of absolute normalized autocorrelations for lags 1..maxlag for each measurement
    Lower is better (0 means perfectly white).
    """
    N, m = innov.shape
    metric = 0.0
    for j in range(m):
        series = innov[:, j] - np.mean(innov[:, j])
        var = np.var(series)
        if var < 1e-12:
            continue
        acf = []
        for lag in range(1, maxlag + 1):
            # unbiased autocorrelation
            r = np.correlate(series[:-lag], series[lag:])[0] / (N - lag)
            acf.append(abs(r / var))
        metric += np.sum(acf)
    # normalize by number of measurement channels
    return metric / m

def tune_filters(Y, t, x0, P0, Q_base, R_base, method='ekf'):
    """
    Simple grid search to scale Q and R by scalar multipliers to minimize whiteness metric.
    Returns best scalars q_scale, r_scale and corresponding outputs.
    """
    q_scales = np.logspace(-3, 3, 13)   # coarse
    r_scales = np.logspace(-3, 3, 13)
    best = (1e9, None, None, None)
    for qs in q_scales:
        for rs in r_scales:
            Q_try = Q_base * qs
            R_try = R_base * rs
            if method == 'ekf':
                x_est, P_est, innov = run_ekf(Y, t, x0, P0, Q_try, R_try)
            else:
                x_est, P_est, innov = run_ukf(Y, t, x0, P0, Q_try, R_try)
            metric = whiteness_metric(innov, maxlag=20)
            if metric < best[0]:
                best = (metric, qs, rs, (x_est, P_est, innov))
    # refine around best
    best_metric, best_qs, best_rs, best_out = best

    if best_metric is None or best_qs is None or best_rs is None or best_out is None:
        return None, None, None, None

    q_scales_fine = np.linspace(best_qs/3, best_qs*3, 13)
    r_scales_fine = np.linspace(best_rs/3, best_rs*3, 13)
    for qs in q_scales_fine:
        for rs in r_scales_fine:
            Q_try = Q_base * qs
            R_try = R_base * rs
            if method == 'ekf':
                x_est, P_est, innov = run_ekf(Y, t, x0, P0, Q_try, R_try)
            else:
                x_est, P_est, innov = run_ukf(Y, t, x0, P0, Q_try, R_try)
            metric = whiteness_metric(innov, maxlag=20)
            if metric < best_metric:
                best_metric, best_qs, best_rs, best_out = metric, qs, rs, (x_est, P_est, innov)
    return best_qs, best_rs, best_metric, best_out

# Run tuning for EKF (this will take time). Do UKF tuning similarly.
print("Tuning EKF (coarse grid + refine)...")
best_qs_ekf, best_rs_ekf, best_metric_ekf, best_out_ekf = tune_filters(Y, t, x0, P0, Q_base, R_base, method='ekf')
print(f"EKF best q_scale={best_qs_ekf}, r_scale={best_rs_ekf}, metric={best_metric_ekf}")

print("Tuning UKF (coarse grid + refine)...")
best_qs_ukf, best_rs_ukf, best_metric_ukf, best_out_ukf = tune_filters(Y, t, x0, P0, Q_base, R_base, method='ukf')
print(f"UKF best q_scale={best_qs_ukf}, r_scale={best_rs_ukf}, metric={best_metric_ukf}")
